# Multi-Turn Evaluation & Conversation Simulation with MLflow

This notebook demonstrates MLflow's multi-turn evaluation capabilities (introduced in MLflow 3.10) for assessing conversational AI quality across entire sessions — not just individual turns.

**What you'll learn:**
1. Evaluate existing conversations with session-level scorers
2. Simulate conversations to test agent versions with `ConversationSimulator`
3. Use built-in multi-turn judges (`ConversationCompleteness`, `UserFrustration`, `KnowledgeRetention`)
4. Create custom multi-turn judges with `{{ conversation }}` template variable

**Prerequisites:**
- A SageMaker MLflow Tracking Server
- A Strands Agent deployed on AgentCore Runtime (see `deploy_to_agentcore.py`)
- AWS credentials with Bedrock + SageMaker + AgentCore access
- Python 3.10+, MLflow >= 3.10

## 0. Install Dependencies

In [ ]:
%pip install --quiet "mlflow>=3.10" boto3 pandas sagemaker-mlflow

## 1. Configuration

In [ ]:
import json
import os

# Fill here the region and the Bedrock Model ID
REGION = "us-west-2"
BEDROCK_MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"
JUDGE_MODEL = f"bedrock:/{BEDROCK_MODEL_ID}"

# Load AgentCore config (created by deploy_to_agentcore.py)
CONFIG_PATH = os.path.join(os.getcwd(), "agentcore_config.json")
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        ac_config = json.load(f)
    AGENT_RUNTIME_ARN = ac_config["agent_runtime_arn"]
else:
    # Set this manually if config file is not available
    AGENT_RUNTIME_ARN = "YOUR_AGENT_RUNTIME_ARN_HERE"

print(f"AgentCore Runtime ARN: {AGENT_RUNTIME_ARN}")

## 2. Connect to SageMaker MLflow Tracking Server

In [ ]:
import boto3
import mlflow

# Fill here your MLFLOW Aplication ARN
TRACKING_URI = "YOUR_MLFLOW_APP_ARN"
EXP_NAME = "multi-turn-eval-agentcore"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXP_NAME)
mlflow.bedrock.autolog()

# AgentCore client for invoking the deployed agent
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

In [ ]:
# Pull credentials from the IAM role and pass them explicitly
credentials = boto3.Session().get_credentials().get_frozen_credentials()

os.environ["AWS_ACCESS_KEY_ID"] = credentials.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = credentials.secret_key
if credentials.token:
    os.environ["AWS_SESSION_TOKEN"] = credentials.token

## 3. Define a Conversational Agent (via AgentCore)

Instead of calling `bedrock.converse()` directly, we invoke the Strands Agent deployed on AgentCore Runtime. Each call is traced and tagged with a `session` ID so MLflow can group turns into conversations.

In [ ]:
import json


def invoke_agentcore(prompt: str, session_id: str) -> str:
    """Invoke the Strands Agent on AgentCore Runtime."""
    payload = json.dumps({"prompt": prompt}).encode()
    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=payload,
        qualifier="DEFAULT",
    )
    body = response["response"].read()
    data = json.loads(body)
    return data.get("response", data.get("result", str(data)))


@mlflow.trace
def chat_agent(messages: list[dict], session_id: str) -> str:
    """Multi-turn chat agent via AgentCore. Accepts conversation history, returns assistant response."""
    # Tag the trace with session ID for multi-turn grouping
    mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})

    # Build the prompt from the last user message (AgentCore handles single prompts)
    # Include conversation context in the prompt for multi-turn awareness
    if len(messages) > 1:
        context_lines = []
        for m in messages[:-1]:
            role = m["role"].capitalize()
            context_lines.append(f"{role}: {m['content']}")
        context = "\n".join(context_lines)
        last_msg = messages[-1]["content"]
        prompt = f"Previous conversation:\n{context}\n\nUser: {last_msg}"
    else:
        prompt = messages[-1]["content"]

    return invoke_agentcore(prompt, session_id)

## 4. Generate Pre-Recorded Conversations

We'll run two multi-turn conversations to create traced sessions, then evaluate them with session-level scorers.

In [ ]:
import uuid


def run_conversation(turns: list[str], session_id: str) -> list[dict]:
    """Run a multi-turn conversation, returning the full message history."""
    history = []
    for user_msg in turns:
        history.append({"role": "user", "content": user_msg})
        reply = chat_agent(history, session_id=session_id)
        history.append({"role": "assistant", "content": reply})
        print(f"  User: {user_msg}")
        print(f"  Agent: {reply[:120]}...\n")
    return history


# Conversation 1: Smooth, goal-oriented
session_1 = f"session-{uuid.uuid4().hex[:8]}-agentcore-padding"
print(f"=== Conversation 1 ({session_1}) ===")
run_conversation(
    turns=[
        "What is MLflow experiment tracking?",
        "How do I log parameters and metrics in a run?",
        "Can I compare runs across experiments?",
    ],
    session_id=session_1,
)

# Conversation 2: Potentially frustrating — agent may lose context
session_2 = f"session-{uuid.uuid4().hex[:8]}-agentcore-padding"
print(f"=== Conversation 2 ({session_2}) ===")
run_conversation(
    turns=[
        "I'm deploying a model to SageMaker but getting an error.",
        "I already told you it's a SageMaker deployment error. Can you help or not?",
        "Fine, what about using MLflow with SageMaker instead?",
    ],
    session_id=session_2,
)

## 5. Evaluate Pre-Generated Conversations with Session-Level Scorers

Retrieve the traced conversations and evaluate them with MLflow's built-in multi-turn judges.

| Judge | What it evaluates |
|---|---|
| `ConversationCompleteness` | Were all user questions addressed by the end? |
| `UserFrustration` | Did the user become frustrated? Was it resolved? |
| `KnowledgeRetention` | Does the agent remember info from earlier turns? |

In [ ]:
from mlflow.genai.scorers import (
    ConversationCompleteness,
    UserFrustration,
    KnowledgeRetention,
)

# Retrieve all traces from this experiment
experiment = mlflow.get_experiment_by_name(EXP_NAME)
traces = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    return_type="list",
)

print(f"Retrieved {len(traces)} traces across sessions")

# Evaluate — MLflow automatically groups traces by session ID
results = mlflow.genai.evaluate(
    data=traces,
    scorers=[
        ConversationCompleteness(model=JUDGE_MODEL),
        UserFrustration(model=JUDGE_MODEL),
        KnowledgeRetention(model=JUDGE_MODEL),
    ],
)

print("\n=== Session-Level Metrics ===")
for metric, value in results.metrics.items():
    print(f"  {metric}: {value}")

In [ ]:
results.result_df

## 6. Custom Multi-Turn Judge

Create a custom judge using `make_judge` with the `{{ conversation }}` template variable. This injects the full conversation history for the judge to analyze.

In [ ]:
from typing import Literal
from mlflow.genai.judges import make_judge

tone_consistency_judge = make_judge(
    name="tone_consistency",
    instructions=(
        "Analyze the {{ conversation }} and evaluate whether the assistant "
        "maintains a consistent, professional, and helpful tone throughout "
        "all turns — even when the user is frustrated or unclear. "
        "Rate as 'consistent', 'mostly_consistent', or 'inconsistent'."
    ),
    feedback_value_type=Literal["consistent", "mostly_consistent", "inconsistent"],
    model=JUDGE_MODEL,
)

# Evaluate with the custom judge
custom_results = mlflow.genai.evaluate(
    data=traces,
    scorers=[tone_consistency_judge],
)

print("=== Tone Consistency Results ===")
for metric, value in custom_results.metrics.items():
    print(f"  {metric}: {value}")

## 7. Conversation Simulation with `ConversationSimulator`

Instead of manually crafting conversations, define **test scenarios** with goals and personas. MLflow simulates realistic user interactions with your agent and evaluates the results.

**Key difference:** The `predict_fn` now invokes the Strands Agent on AgentCore Runtime instead of calling `bedrock.converse()` directly.

In [ ]:
from mlflow.genai.simulators import ConversationSimulator
from mlflow.genai.scorers import ConversationCompleteness, UserFrustration

# Define test scenarios
test_cases = [
    {
        "goal": (
            "Learn how to set up MLflow experiment tracking with SageMaker. "
            "The agent should provide code examples and explain the key concepts."
        ),
    },
    {
        "goal": "Debug why model artifacts are not being logged to the tracking server.",
        "persona": "You are a frustrated data scientist who has been stuck on this for hours.",
    },
    {
        "goal": "Understand the difference between MLflow experiments, runs, and registered models.",
        "persona": "You are a beginner who needs step-by-step explanations.",
        "simulation_guidelines": [
            "Start with a broad question before asking about specifics.",
            "Do not mention registered models until the assistant brings them up.",
        ],
    },
]

simulator = ConversationSimulator(
    test_cases=test_cases,
    max_turns=4,
    user_model=JUDGE_MODEL,  # Use Bedrock to simulate the user too
)

### Define the agent predict function

The simulator calls this function with the conversation history (list of message dicts) at each turn. Instead of calling Bedrock directly, we invoke the AgentCore-deployed Strands Agent.

In [ ]:
def predict_fn(input: list[dict], **kwargs) -> str:
    """Agent function for the simulator. Invokes AgentCore Runtime."""
    session_id = kwargs.get("mlflow_session_id", "sim-unknown")
    # Ensure session_id is ≥33 chars (AgentCore requirement)
    if len(session_id) < 33:
        session_id = session_id + "-" + "x" * (33 - len(session_id) - 1)

    # Build a single prompt with conversation context
    if len(input) > 1:
        context_lines = []
        for m in input[:-1]:
            role = m["role"].capitalize()
            context_lines.append(f"{role}: {m['content']}")
        context = "\n".join(context_lines)
        last_msg = input[-1]["content"]
        prompt = f"Previous conversation:\n{context}\n\nUser: {last_msg}"
    else:
        prompt = input[-1]["content"]

    # Tag trace with session for multi-turn grouping
    with mlflow.start_span(name="chat_agent") as span:
        mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
        return invoke_agentcore(prompt, session_id)

### Run simulation + evaluation in one call

`mlflow.genai.evaluate()` accepts the simulator as `data`, runs the conversations, and scores them — all in one step.

In [ ]:
sim_results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=predict_fn,
    scorers=[
        ConversationCompleteness(model=JUDGE_MODEL),
        UserFrustration(model=JUDGE_MODEL),
        tone_consistency_judge,  # Our custom judge from earlier
    ],
)

print("=== Simulation Evaluation Metrics ===")
for metric, value in sim_results.metrics.items():
    print(f"  {metric}: {value}")

In [ ]:
sim_results.result_df

## 8. Best Practices for Conversational AI Quality Assurance

### Evaluation Strategy

| Stage | Approach | Scorers |
|---|---|---|
| **Development** | Simulate conversations with diverse personas | `ConversationCompleteness`, `UserFrustration`, custom judges |
| **Pre-release** | Replay production-derived test cases against new version | `KnowledgeRetention`, `ConversationalRoleAdherence` |
| **Production** | Evaluate live sessions continuously | `UserFrustration`, `ConversationalSafety` |

### Key Recommendations

1. **Tag every trace with a session ID** — use `mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})` so MLflow can group turns
2. **Combine single-turn and multi-turn scorers** — single-turn catches per-response issues (safety, relevance); multi-turn catches session-level problems (frustration, completeness)
3. **Use `simulation_guidelines`** to reproduce specific failure patterns observed in production
4. **Extract test cases from production** with `generate_test_cases()` to create regression tests from real conversations
5. **Version your test scenarios** as MLflow Evaluation Datasets for reproducible comparisons across agent versions
6. **Run simulations in CI/CD** to catch regressions before deployment
7. **Deploy agents to AgentCore** for managed scaling, and test the deployed version (not just local) to catch deployment-specific issues

## Next Steps

- **View results** in the MLflow UI → experiment `multi-turn-eval-agentcore` → Sessions tab
- **Add more built-in judges**: `ConversationalGuidelines`, `ConversationalRoleAdherence`, `ConversationalSafety`, `ConversationalToolCallEfficiency`
- **Extract test cases from production**: use `mlflow.genai.simulators.generate_test_cases(sessions)` to create scenarios from real conversations
- **Persist test cases**: save to MLflow Evaluation Datasets with `create_dataset()` for reproducible testing
- **Set up automatic evaluation**: configure MLflow to run judges automatically on new traces as they're logged